In [84]:
import json
import pandas as pd
from dotenv import load_dotenv
import os
import aisuite as ai
import sys

In [85]:
project_root = "/home/mohammad/Documents/Codes/agentic_codes"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [86]:
from src.utils import db_helpers as utils

In [87]:
load_dotenv()

True

In [88]:
client = ai.Client()

In [89]:
utils.create_transactions_db()
utils.print_html(utils.get_schema(f"{project_root}/data/products.db"))

SQLite database 'products.db' created with a single 'transactions' table (event-sourced).


In [90]:
def generate_sql_code(
    question: str,
    schema: str,
    model: str
) -> str:
    prompt = f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """

    response = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}]
    )
    
    return response.choices[0].message.content

In [91]:
def refine_sql_code(
    question: str,
    schema: str,
    model: str,
    sql_v1_output: str,
    executed_sql_v1_output: str
):
    prompt = f"""    
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_v1_output}
    
    Output of executed Original SQL:
    {executed_sql_v1_output}

    Table Schema:
    {schema}
    
    Step 1: Briefly evaluate if the SQL OUTPUT fully answers the user's question.
    Step 2: If improvement is needed, provide a refined SQL query for SQLite.
    If the original SQL is already correct, return it unchanged.
    output must be includes color and sales.
    Return STRICT JSON with two fields:
    {{
    "feedback": "<1-3 sentences explaining the gap or confirming correctness>",
    "refined_sql": "<final SQL to run>"
    }}
    """
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}]
    )
    
    content = response.choices[0].message.content.strip()

    try:
        # Remove Markdown code fences if present
        if content.startswith("```"):
            content = content.split("\n", 1)[1]
            content = content.rsplit("```", 1)[0].strip()

        obj = json.loads(content)

        feedback = str(obj.get("feedback", "")).strip()
        refined_sql = str(obj.get("refined_sql", "")).strip()

        if not refined_sql:
            raise ValueError("Model did not return refined SQL")

    except Exception as e:
        raise ValueError(
            f"Could not parse model response: DIL0\n"
            f"Raw response:\nDIL1"
        )

    return feedback, refined_sql


In [92]:
def run_sql_workflow(
        question: str,
        model: str,
        schema: str
    ):

    sql_v1_out = generate_sql_code(question=question, schema=schema, model=model)
    utils.print_html(sql_v1_out, title="SQL Query 1 output - ❌ this is not final answer")
    
    executed_sql_v1 = utils.execute_sql(sql_v1_out, "products.db")
    utils.print_html(executed_sql_v1, title="SQL Query 1 executer output - ❌ this is not final answer")
    
    feedback, sql_v2_out = refine_sql_code(question=question, schema=schema, model=model, sql_v1_output=sql_v1_out, executed_sql_v1_output=executed_sql_v1)
    utils.print_html(feedback, title="feedback on SQL Query 1 output")
    utils.print_html(sql_v2_out, title="Refined SQL Query")
    
    executed_sql_v2 = utils.execute_sql(sql_v2_out, "products.db")
    utils.print_html(executed_sql_v2, title="SQL Query 2 executer output - ✅ final answer")

In [93]:
schema = """
table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

model = "groq:qwen/qwen3.8-27b"
question = "Which color of product has the highest total sales?"

In [94]:
run_sql_workflow(question=question, model=model,schema=schema)

color,total_sales
blue,-150511.18


color,sales
white,68437.36
